In [0]:
CREATE OR REPLACE TABLE workspace.default.rodent_complaints_clean AS
WITH base AS (
  SELECT
    CAST(unique_key AS STRING)                              AS complaint_id,
    CAST(created_date AS TIMESTAMP)                         AS created_at,
    CAST(closed_date  AS TIMESTAMP)                         AS closed_at,
    status,
    descriptor,
    CASE location_type
      WHEN 'Parking Lot/Garage'       THEN 'Parking Lot or Garage'
      WHEN 'Catch Basin/Sewer'        THEN 'Catch Basin or Sewer'
      WHEN 'Day Care/Nursery'         THEN 'Day Care or Nursery'
      WHEN 'School/Pre-School'        THEN 'School'
      ELSE location_type END                                AS location_type,
    -- the upload UI may type ZIP as a number: 11354.0 or 7307. Strip ".0", re-pad to 5 digits.
    LPAD(regexp_replace(CAST(incident_zip AS STRING), '\\.0$', ''), 5, '0') AS zip,
    INITCAP(borough)                                        AS borough,
    CAST(latitude  AS DOUBLE)                               AS latitude,
    CAST(longitude AS DOUBLE)                               AS longitude
  FROM workspace.default.rat_sightings
)
SELECT
  complaint_id, created_at, closed_at, status, descriptor, location_type, zip, borough, latitude, longitude,
  descriptor IN ('Rat Sighting', 'Signs of Rodents', 'Mouse Sighting')            AS is_sighting,
  (unix_timestamp(closed_at) - unix_timestamp(created_at)) / 3600.0                AS hours_to_close,
  closed_at IS NOT NULL AND unix_timestamp(closed_at) - unix_timestamp(created_at) < 60 AS auto_closed,
  CASE WHEN created_at < TIMESTAMP '2026-04-01' THEN 'auto_close_era' ELSE 'post_reform' END AS era,
  closed_at IS NULL AND datediff(TIMESTAMP '2026-09-17', created_at) > 30           AS open_over_30d,
  COALESCE(CONCAT(CAST(latitude AS STRING), ',', CAST(longitude AS STRING)), complaint_id) AS location_key
FROM base
WHERE zip RLIKE '^1[01][0-9]{3}$' AND zip <> '11701';        -- drops junk ZIP 12345 (1 row). Expect 50,953 rows.

-- ============================================================
-- 2. restaurant_inspections_clean — one row per VIOLATION, NYC restaurants only, exact dups removed
-- ============================================================
CREATE OR REPLACE TABLE workspace.default.restaurant_inspections_clean AS
SELECT DISTINCT
  CAST(camis AS STRING)                                     AS camis,
  dba                                                       AS restaurant_name,
  boro                                                      AS borough,
  LPAD(regexp_replace(CAST(zipcode AS STRING), '\\.0$', ''), 5, '0') AS zip,
  cuisine_description                                       AS cuisine,
  CAST(inspection_date AS DATE)                             AS inspection_date,
  violation_code,
  violation_description,
  critical_flag,
  CAST(score AS INT)                                        AS inspection_score,
  grade,
  violation_code IN ('04K', '04L')                          AS is_rodent_evidence,
  violation_code = '04K'                                    AS is_rat_evidence
FROM workspace.default.restaurant_inspections
WHERE zipcode IS NOT NULL
  AND LPAD(regexp_replace(CAST(zipcode AS STRING), '\\.0$', ''), 5, '0') RLIKE '^1[01][0-9]{3}$'
  AND LPAD(regexp_replace(CAST(zipcode AS STRING), '\\.0$', ''), 5, '0') <> '11701'
  AND CAST(boro AS STRING) <> '0';                           -- Expect 156,300 rows.

-- ============================================================
-- 3. restaurants — one row per restaurant (camis). USE THIS for any "how many restaurants" question.
-- ============================================================
CREATE OR REPLACE TABLE workspace.default.restaurants AS
SELECT
  camis,
  MAX(restaurant_name)                                      AS restaurant_name,
  MAX(borough)                                              AS borough,
  MAX(zip)                                                  AS zip,
  MAX(cuisine)                                              AS cuisine,
  COUNT(DISTINCT inspection_date)                           AS inspections,
  COUNT(violation_code)                                     AS violations,
  bool_or(is_rodent_evidence)                               AS has_rodent_evidence,
  bool_or(is_rat_evidence)                                  AS has_rat_evidence,
  MAX(inspection_date)                                      AS last_inspection_date
FROM workspace.default.restaurant_inspections_clean
GROUP BY camis;                                             -- Expect 25,772 rows; 5,917 with rodent evidence.

SHOW TABLES IN workspace.default;
